# Modelo SIR: teoria na prática (versão simples)

**Como rodar:** use o ambiente do projeto (requirements.txt) e execute as células de cima para baixo. Nenhum dado externo é necessário.

**Referência do artigo:** *(a completar: autores, ano, título, veículo e DOI do artigo de onde vêm as equações)*

## 1. O modelo

A população é dividida em 3 grupos, em **frações** ($S + I + R = 1$):

- **S (suscetíveis):** saudáveis, podem se infectar
- **I (infectados):** estão doentes e transmitem
- **R (removidos):** se curaram (imunes) ou morreram

**Equações do artigo (Eqs. 1 a 3):**

- $dS/dt = -a S I$
- $dI/dt = a S I - b I$
- $dR/dt = b I$

Onde:
- **$a S I$:** novos infectados (encontros entre suscetíveis e infectados)
- **$b I$:** quem sai de infectado para removido

Como trabalhamos com frações, o artigo usa $a N$; no código isso é chamado de **beta**, enquanto **b** mantém o mesmo nome.

**Quantidades-chave do artigo:**

- **$\rho$ (taxa de remoção relativa):** $\rho = b / \beta$, o valor crítico de S
- **$R_0$ (reprodutividade basal):** $R_0 = \beta S_0 / b$, quantos novos infectados 1 infectado gera
- **$I_{max}$ (Eq. 11):** $1 - \rho + \rho \ln(\rho / S_0)$, o pico de infectados
- **$S(\infty)$ (Eq. 13):** $S_0 e^{-(1 - S(\infty)) / \rho}$, quem **nunca** se infecta
- **$p_c$:** $1 - 1 / R_0$, a fração a vacinar para evitar a epidemia (não foi testada aqui)

## 2. O código

A cada pequeno passo de tempo `dt`, uma fração das pessoas muda de grupo. É o **método de Euler**: só um laço `for`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

def simular_sir(beta, b, i0, vacinados=0.0, T=40, dt=0.02):
    """Simula o modelo SIR pelo método de Euler e devolve tempos, S, I e R."""
    s = (1 - i0) * (1 - vacinados)
    i = i0
    r = (1 - i0) * vacinados
    tempos, lista_s, lista_i, lista_r = [0.0], [s], [i], [r]

    for passo in range(int(T / dt)):
        novos_infectados = beta * s * i * dt     # de S para I (Eq. 1)
        novos_removidos  = b * i * dt            # de I para R (Eq. 3)

        s = s - novos_infectados                 # Eq. 1
        i = i + novos_infectados - novos_removidos   # Eq. 2
        r = r + novos_removidos                  # Eq. 3

        tempos.append((passo + 1) * dt)
        lista_s.append(s); lista_i.append(i); lista_r.append(r)

    return np.array(tempos), np.array(lista_s), np.array(lista_i), np.array(lista_r)

## 3. Quando há epidemia?

Para o artigo, **epidemia significa que o número de infectados chega a ficar maior que o inicial** ($I(t) > I_0$ em algum instante).

- Se **$S_0 > \rho$** (ou seja, **$R_0 > 1$**): ocorre epidemia.
- Se **$S_0 < \rho$** (**$R_0 < 1$**): I só diminui e a doença some.

Fixamos `b = 1` e escolhemos `beta` para obter cada $R_0$ desejado.

In [ ]:
b = 1.0
i0 = 0.001

plt.figure(figsize=(11, 3.5))
linhas = []
for R0 in [0.8, 1.5, 3.0]:
    beta = R0 * b / (1 - i0)              # de R0 = beta*S0/b vem beta = R0*b/S0
    t, s, i, r = simular_sir(beta, b, i0)

    plt.subplot(1, 2, 1); plt.plot(t, i, label=f"R0 = {R0}")
    plt.subplot(1, 2, 2); plt.plot(t, 1 - s, label=f"R0 = {R0}")     # 1 - S = quem já se infectou (I + R)

    linhas.append({"R0": R0, "houve epidemia? (I > I0)": bool(i.max() > i0 * 1.001), "pico de I": i.max()})

plt.subplot(1, 2, 1); plt.title("Infectados I(t)"); plt.xlabel("tempo"); plt.legend()
plt.subplot(1, 2, 2); plt.title("Já infectados (I + R)"); plt.xlabel("tempo"); plt.legend()
plt.tight_layout(); plt.show()
pd.DataFrame(linhas).round(3)

## 4. Conferindo as fórmulas do artigo

Comparamos o que a **fórmula** prevê com o que a **simulação** deu: o pico $I_{max}$ (Eq. 11) e quem escapa, $S(\infty)$ (Eq. 13).

Como $S(\infty)$ aparece dos dois lados da Eq. 13 (equação transcendental), resolvemos por repetição: começamos com um chute e aplicamos a fórmula várias vezes até estabilizar.

In [ ]:
linhas = []
for R0 in [1.5, 2.0, 3.0, 5.0]:
    beta = R0 * b / (1 - i0)
    rho = b / beta
    s0 = 1 - i0

    # fórmulas do artigo
    i_max_formula = 1 - rho + rho * np.log(rho / s0)            # Eq. 11
    s_inf = 0.0
    for _ in range(5000):                                       # Eq. 13: s = s0 * exp(-(1-s)/rho)
        s_inf = s0 * np.exp(-(1 - s_inf) / rho)
    i_total_formula = i0 + s0 - s_inf                           # Eq. 14

    # simulação
    t, s, i, r = simular_sir(beta, b, i0, T=80)

    linhas.append({"R0": R0,
                   "I_max fórmula": i_max_formula, "I_max simulado": i.max(),
                   "S(inf) fórmula": s_inf,        "S(inf) simulado": s[-1],
                   "I_total fórmula": i_total_formula, "I_total simulado": i0 + s0 - s[-1]})
pd.DataFrame(linhas).round(3)

## 5. Plano de fase (Figura 2 do artigo)

Cada curva mostra I contra S ao longo do tempo para uma condição inicial diferente (todas começam na reta $S_0 + I_0 = 1$). A linha tracejada é **$\rho$**.

- Começa à **direita** de $\rho$ ($S_0 > \rho$): I **sobe** e depois cai, então há **epidemia** (vermelho).
- Começa à **esquerda** de $\rho$ ($S_0 < \rho$): I só **cai**, então não há epidemia (azul).

O artigo também mostra que $I + S - \rho \ln(S)$ é **constante** em cada trajetória (Eq. 9), o que verificamos abaixo.

In [ ]:
R0_ref = 2.0
beta = R0_ref * b
rho = b / beta

plt.figure(figsize=(6.5, 4.2))
for s0 in [0.95, 0.8, 0.65, 0.4, 0.3, 0.2, 0.1]:
    i0_ = 1 - s0
    # simulação genérica com beta, b fixos e s0 escolhido:
    s, i = s0, i0_
    trajeto_s, trajeto_i = [s], [i]
    for _ in range(4000):
        novos = beta * s * i * 0.02
        s, i = s - novos, i + novos - b * i * 0.02
        trajeto_s.append(s); trajeto_i.append(i)
    trajeto_s, trajeto_i = np.array(trajeto_s), np.array(trajeto_i)

    cor = "red" if s0 > rho else "tab:blue"
    plt.plot(trajeto_s, trajeto_i, color=cor)

    # Eq. 9: I + S - rho*ln(S) deve ser constante
    constante = trajeto_i + trajeto_s - rho * np.log(trajeto_s)
    print(f"S0={s0:.2f}  variação da constante da Eq. 9 ao longo do tempo: {constante.max() - constante.min():.4f}")

plt.plot([0, 1], [1, 0], "k-", lw=1)                 # reta S + I = 1
plt.axvline(rho, ls="--", color="k"); plt.text(rho + 0.01, 0.8, r"$\rho$")
plt.xlabel("S (suscetíveis)"); plt.ylabel("I (infectados)")
plt.title("Plano de fase: vermelho = epidemia, azul = sem epidemia")
plt.xlim(0, 1); plt.ylim(0, 1); plt.show()